# Budgerigar：局部声学编码 + 双 CTC 内容记忆

本轮不再延长旧结构训练。因果局部编码器学习音素邻域，声学序列 CTC 直接约束可辨文字，token bank CTC 约束长程记忆不丢内容；InfoNCE 只作辅助。模型仍是连续神经计算，不设置监听、结束、朗读等离散状态。

In [ ]:
#@title 1. 更新项目与安装
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [key for key in list(sys.modules) if key=='budgerigar' or key.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. 挂载 Drive 并定位特征
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
STATS_PATH=WORK_ROOT/'features'/f'stats.smoke64.arctic_slt.{FEATURE_FINGERPRINT}.pt'
assert FEATURE_MANIFEST.is_file(),FEATURE_MANIFEST
assert STATS_PATH.is_file(),STATS_PATH
import torch,json
stats=torch.load(STATS_PATH,map_location='cpu',weights_only=True)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
#@title 3. 参数规模与单批形状检查
from budgerigar.content_data import CharacterVocabulary,ContentFeatureDataset,collate_content
from budgerigar.content_memory import ContentMemoryConfig,create_content_memory
vocab=CharacterVocabulary()
model_config=ContentMemoryConfig(hidden_dim=192,token_slots=160,update_stride=4,vocabulary_size=len(vocab.symbols),sequence_contrastive=True,local_encoder_layers=4,local_kernel_size=5,acoustic_ctc=True)
model=create_content_memory(model_config)
parameters=sum(p.numel() for p in model.parameters())
print(f'parameters: {parameters:,} ({parameters/1e6:.2f}M)')
preview=ContentFeatureDataset(FEATURE_MANIFEST,'train',stats,vocab,max_records=2,preload=False,update_stride=model_config.update_stride,token_slots=model_config.token_slots)
batch=collate_content([preview[0],preview[1]])
with torch.no_grad(): result=model(batch[0],batch[1],batch[2],batch[3])
print('bank logits:',result[0].shape,'acoustic logits:',result[4]['acoustic_logits'].shape)
assert parameters < 10_000_000

In [ ]:
#@title 4. 旧双 CTC 实验已停止：只读取报告
RUN_DIR=WORK_ROOT/'checkpoints'/f'content_local_dual_ctc_{FEATURE_FINGERPRINT}'
REPORT_PATH=RUN_DIR/'training_report.json'
assert REPORT_PATH.is_file(),REPORT_PATH
report=json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print('该结构已判定平台期；本单元不会继续训练。')
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 5. 独立门槛检查与元数据
history=report['history']
best=min(history,key=lambda row:row['selection_score'])
dual_ctc_pass=best['validation_cer']<0.65 and best['validation_acoustic_cer']<0.65 and best['validation_retrieval_top1']>0.5
print('best:',json.dumps(best,ensure_ascii=False,indent=2))
print('dual_ctc_pass =',dual_ctc_pass)
from budgerigar.experiment import write_run_metadata
metadata=write_run_metadata(RUN_DIR/'run_metadata.json',FEATURE_MANIFEST,{'architecture':'content_local_dual_ctc_memory','parameters':parameters,'best_validation_cer':report['best_validation_cer'],'dual_ctc_pass':dual_ctc_pass},repository=REPO_DIR)
print(metadata.read_text(encoding='utf-8'))